# Indian Share Market Analysis

This notebook demonstrates how to fetch and analyze Indian stock market data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

# For NSE specific data
try:
    from nsepy import get_history
    from nsetools import Nse
except ImportError:
    print('Install NSE tools with: pip install nsepy nsetools')

# Set plotting style
plt.style.use('fivethirtyeight')
%matplotlib inline

## Fetching Stock Data

Let's fetch data for some major Indian stocks.

In [ ]:
# Define stock symbols (NSE tickers usually end with .NS)
stocks = ['RELIANCE.NS', 'TCS.NS', 'HDFCBANK.NS', 'INFY.NS', 'ICICIBANK.NS']

# Fetch data for the last year
data = yf.download(stocks, period='1y')

# Display the first few rows of the closing prices
data['Close'].head()

In [1]:
# -----------------------------
# IMPORTS
# -----------------------------
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# -----------------------------
# DATA FETCH (update dates/tickers)
# -----------------------------
tickers = ["^NSEI", "^GSPC", "^IXIC", "^DJI", "^INDIAVIX"]
raw_data = yf.download(tickers, start="2020-01-01", end="2026-03-05")["Close"]

data = raw_data.rename(columns={
    "^NSEI": "NIFTY",
    "^GSPC": "SP500",
    "^IXIC": "NASDAQ",
    "^DJI": "DOW",
    "^INDIAVIX": "INDIA_VIX"
})

# -----------------------------
# FEATURE ENGINEERING
# -----------------------------
data["NIFTY_mom5"] = data["NIFTY"].pct_change(5)
for col in ["SP500","NASDAQ","DOW","INDIA_VIX"]:
    data[f"{col}_lag1"] = data[col].pct_change(1)
data["VIX_lag1"] = data["INDIA_VIX"].shift(1)
data["VIX_ma20"] = data["INDIA_VIX"].rolling(20).mean()

data["Return"] = data["NIFTY"].pct_change()
data["Target"] = np.where(data["Return"].shift(-1) > 0, 1, 0)

data = data.dropna()
features = ["NIFTY_mom5","SP500_lag1","NASDAQ_lag1","DOW_lag1","VIX_lag1","VIX_ma20"]

# -----------------------------
# TRAIN/TEST SPLIT
# -----------------------------
split = int(len(data) * 0.7)
X = data[features]
y = data["Target"]

X_train = X.iloc[:split]
X_test = X.iloc[split:]
y_train = y.iloc[:split]
y_test = y.iloc[split:]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# -----------------------------
# MODEL TRAINING
# -----------------------------
rf = RandomForestClassifier(n_estimators=300, max_depth=5, random_state=42)
xgb = XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05,
                    subsample=0.8, colsample_bytree=0.8, random_state=42)

rf.fit(X_train, y_train)
xgb.fit(X_train, y_train)

# -----------------------------
# ENSEMBLE PREDICTION
# -----------------------------
rf_prob = rf.predict_proba(X_test)[:,1]
xgb_prob = xgb.predict_proba(X_test)[:,1]

ensemble_prob = (rf_prob + xgb_prob)/2

# Threshold optimization
threshold = 0.53
positions = np.where(ensemble_prob > threshold, 1, -1)

# -----------------------------
# STRATEGY RISK MANAGEMENT
# -----------------------------
cost = 0.0005  # transaction cost
stop_loss = 0.01
take_profit = 0.02

test_returns = data["Return"].iloc[split:].values
strategy_returns = []
position_size = 1.0

for i in range(len(positions)):
    ret = positions[i] * test_returns[i] * position_size
    # Stop-loss / take-profit
    if ret < -stop_loss:
        ret = -stop_loss
    if ret > take_profit:
        ret = take_profit
    strategy_returns.append(ret)

strategy_returns = np.array(strategy_returns)

# Adjust for transaction costs
trades = np.abs(np.diff(positions))
strategy_returns[1:] -= trades * cost

cumulative = np.cumsum(strategy_returns)
num_trades = np.sum(trades)

# Sharpe ratio
mean_ret = np.mean(strategy_returns)
std_ret = np.std(strategy_returns)
sharpe = (mean_ret / std_ret) * np.sqrt(252)

# -----------------------------
# RESULTS
# -----------------------------
print("RandomForest Accuracy:", rf.score(X_test, y_test))
print("XGBoost Accuracy:", xgb.score(X_test, y_test))
ensemble_pred = np.where(ensemble_prob > threshold, 1, 0)
ensemble_acc = np.mean(ensemble_pred == y_test)
print("Ensemble Accuracy:", ensemble_acc)
print("Best Threshold:", threshold)
print("Total Return:", cumulative[-1])
print("Return After Cost:", np.sum(strategy_returns))
print("Sharpe Ratio:", sharpe)
print("Number of Trades:", num_trades)

[*********************100%***********************]  5 of 5 completed
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4048\2576110892.py:29: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  data["NIFTY_mom5"] = data["NIFTY"].pct_change(5)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4048\2576110892.py:31: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  data[f"{col}_lag1"] = data[col].pct_change(1)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4048\2576110892.py:31: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-l

RandomForest Accuracy: 0.5901639344262295
XGBoost Accuracy: 0.5491803278688525
Ensemble Accuracy: 0.5901639344262295
Best Threshold: 0.53
Total Return: 0.06135973949716585
Return After Cost: 0.06135973949716585
Sharpe Ratio: 1.307372825831857
Number of Trades: 92


In [2]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

# =========================
# 1. Data Download
# =========================
tickers = ["^NSEI", "^GSPC", "^IXIC", "^DJI", "^INDIAVIX"]
raw_data = yf.download(tickers, start="2023-01-01", end="2026-03-05")
data = raw_data['Close']  # Use Close prices

data = data.rename(columns={
    "^NSEI": "NIFTY",
    "^GSPC": "SP500",
    "^IXIC": "NASDAQ",
    "^DJI": "DOW",
    "^INDIAVIX": "INDIA_VIX"
})

# =========================
# 2. Feature Engineering
# =========================
data['NIFTY_mom5'] = data['NIFTY'].pct_change(5)
for col in ['SP500', 'NASDAQ', 'DOW', 'INDIA_VIX']:
    data[f'{col}_lag1'] = data[col].shift(1)

data['VIX_ma20'] = data['INDIA_VIX'].rolling(20).mean()
data['Return'] = data['NIFTY'].pct_change()
data['Target'] = (data['Return'] > 0).astype(int)
data.dropna(inplace=True)

features = ["NIFTY_mom5","SP500_lag1","NASDAQ_lag1","DOW_lag1","INDIA_VIX_lag1","VIX_ma20"]
X = data[features].values
y = data["Target"].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

# =========================
# 3. Rolling Window Backtest
# =========================
window_size = 500  # training window
step_size = 20     # move forward by 20 days
threshold_history = []
returns_history = []

for start in range(0, len(data) - window_size, step_size):
    end = start + window_size
    X_train, y_train = X[start:end], y[start:end]
    X_test, y_test = X[end:end+step_size], y[end:end+step_size]
    
    if len(X_test) == 0:
        break

    # -----------------
    # 3a. Hyperparameter tuned RandomForest
    # -----------------
    rf_params = {
        'n_estimators': [100, 200],
        'max_depth': [3, 5, 7],
        'random_state': [42]
    }
    rf_grid = GridSearchCV(RandomForestClassifier(), rf_params, cv=3)
    rf_grid.fit(X_train, y_train)
    rf_best = rf_grid.best_estimator_
    
    # -----------------
    # 3b. Hyperparameter tuned XGBoost
    # -----------------
    xgb_params = {
        'n_estimators': [100, 200],
        'max_depth': [3, 5],
        'learning_rate': [0.05, 0.1],
        'subsample': [0.8, 1],
        'colsample_bytree': [0.8, 1]
    }
    xgb_grid = GridSearchCV(XGBClassifier(use_label_encoder=False, eval_metric='logloss'), xgb_params, cv=3)
    xgb_grid.fit(X_train, y_train)
    xgb_best = xgb_grid.best_estimator_
    
    # -----------------
    # 3c. Ensemble prediction
    # -----------------
    rf_prob = rf_best.predict_proba(X_test)[:,1]
    xgb_prob = xgb_best.predict_proba(X_test)[:,1]
    ensemble_prob = (rf_prob + xgb_prob) / 2

    # -----------------
    # 3d. Dynamic threshold
    # Threshold = mean(probabilities in training set)
    threshold = rf_best.predict_proba(X_train)[:,1].mean()  # simple dynamic
    threshold_history.append(threshold)

    positions = np.where(ensemble_prob > threshold, 1, -1)

    # -----------------
    # 3e. Position sizing: confidence-based
    # size = abs(prob - threshold)
    sizes = np.abs(ensemble_prob - threshold)
    strategy_returns = positions * sizes * data['Return'].iloc[end:end+step_size].values

    returns_history.extend(strategy_returns)

# =========================
# 4. Performance Metrics
# =========================
returns_history = np.array(returns_history)
total_return = np.sum(returns_history)
cost = 0.0005
trades = np.sum(np.abs(np.diff(np.sign(returns_history))))
return_after_cost = total_return - trades * cost

mean_ret = np.mean(returns_history)
std_ret = np.std(returns_history)
sharpe = (mean_ret / std_ret) * np.sqrt(252) if std_ret != 0 else 0

print("Total Return:", total_return)
print("Return After Cost:", return_after_cost)
print("Sharpe Ratio:", sharpe)
print("Number of Trades:", trades)

[*********************100%***********************]  5 of 5 completed

Total Return: 0.0
Return After Cost: 0.0
Sharpe Ratio: nan
Number of Trades: 0.0



C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4048\1512073225.py:27: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  data['NIFTY_mom5'] = data['NIFTY'].pct_change(5)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4048\1512073225.py:32: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  data['Return'] = data['NIFTY'].pct_change()
C:\ProgramData\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\ProgramData\anaconda3\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in s

In [3]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

# =========================
# 1. Data Download
# =========================
tickers = ["^NSEI", "^GSPC", "^IXIC", "^DJI", "^INDIAVIX"]
raw_data = yf.download(tickers, start="2023-01-01", end="2026-03-05")
data = raw_data['Close']  # Use Close prices

data = data.rename(columns={
    "^NSEI": "NIFTY",
    "^GSPC": "SP500",
    "^IXIC": "NASDAQ",
    "^DJI": "DOW",
    "^INDIAVIX": "INDIA_VIX"
})

# =========================
# 2. Feature Engineering
# =========================
data['NIFTY_mom5'] = data['NIFTY'].pct_change(5)
for col in ['SP500', 'NASDAQ', 'DOW', 'INDIA_VIX']:
    data[f'{col}_lag1'] = data[col].shift(1)

data['VIX_ma20'] = data['INDIA_VIX'].rolling(20).mean()
data['Return'] = data['NIFTY'].pct_change()
data['Target'] = (data['Return'] > 0).astype(int)
data.dropna(inplace=True)

features = ["NIFTY_mom5","SP500_lag1","NASDAQ_lag1","DOW_lag1","INDIA_VIX_lag1","VIX_ma20"]
X = data[features].values
y = data["Target"].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

# =========================
# 3. Rolling Window Backtest with Ensemble + SL/PT
# =========================
window_size = 500
step_size = 20
threshold_history = []
returns_history = []

stop_loss = 0.01  # 1%
profit_target = 0.02  # 2%

for start in range(0, len(data) - window_size, step_size):
    end = start + window_size
    X_train, y_train = X[start:end], y[start:end]
    X_test, y_test = X[end:end+step_size], y[end:end+step_size]
    
    if len(X_test) == 0:
        break

    # ---- Hyperparameter tuned RandomForest ----
    rf_params = {'n_estimators':[100,200], 'max_depth':[3,5,7], 'random_state':[42]}
    rf_grid = GridSearchCV(RandomForestClassifier(), rf_params, cv=3)
    rf_grid.fit(X_train, y_train)
    rf_best = rf_grid.best_estimator_

    # ---- Hyperparameter tuned XGBoost ----
    xgb_params = {
        'n_estimators':[100,200], 'max_depth':[3,5],
        'learning_rate':[0.05,0.1], 'subsample':[0.8,1],
        'colsample_bytree':[0.8,1]
    }
    xgb_grid = GridSearchCV(XGBClassifier(use_label_encoder=False, eval_metric='logloss'), xgb_params, cv=3)
    xgb_grid.fit(X_train, y_train)
    xgb_best = xgb_grid.best_estimator_

    # ---- Ensemble Probabilities ----
    rf_prob = rf_best.predict_proba(X_test)[:,1]
    xgb_prob = xgb_best.predict_proba(X_test)[:,1]
    ensemble_prob = (rf_prob + xgb_prob) / 2

    # ---- Dynamic Threshold ----
    threshold = rf_best.predict_proba(X_train)[:,1].mean()
    threshold_history.append(threshold)

    positions = np.where(ensemble_prob > threshold, 1, -1)
    sizes = np.abs(ensemble_prob - threshold)

    # ---- Apply Stop-loss and Profit-target ----
    test_returns = data['Return'].iloc[end:end+step_size].values
    strategy_returns = []

    for pos, size, ret in zip(positions, sizes, test_returns):
        pnl = pos * size * ret

        # Check stop-loss
        if pnl < -stop_loss:
            pnl = -stop_loss
        # Check profit-target
        if pnl > profit_target:
            pnl = profit_target

        strategy_returns.append(pnl)

    returns_history.extend(strategy_returns)

# =========================
# 4. Performance Metrics
# =========================
returns_history = np.array(returns_history)
total_return = np.sum(returns_history)
cost = 0.0005
trades = np.sum(np.abs(np.diff(np.sign(returns_history))))
return_after_cost = total_return - trades * cost

mean_ret = np.mean(returns_history)
std_ret = np.std(returns_history)
sharpe = (mean_ret / std_ret) * np.sqrt(252) if std_ret != 0 else 0

print("Total Return:", total_return)
print("Return After Cost:", return_after_cost)
print("Sharpe Ratio:", sharpe)
print("Number of Trades:", trades)

[*********************100%***********************]  5 of 5 completed

Total Return: 0.0
Return After Cost: 0.0
Sharpe Ratio: nan
Number of Trades: 0.0



C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4048\2431468566.py:27: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  data['NIFTY_mom5'] = data['NIFTY'].pct_change(5)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4048\2431468566.py:32: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  data['Return'] = data['NIFTY'].pct_change()
C:\ProgramData\anaconda3\Lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\ProgramData\anaconda3\Lib\site-packages\numpy\_core\_methods.py:144: RuntimeWarning: invalid value encountered in s

In [4]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
plt.plot(cumulative_returns, label="Strategy")
plt.plot(cumulative_returns_benchmark, label="Benchmark", alpha=0.7)
plt.title("Cumulative Returns")
plt.xlabel("Date")
plt.ylabel("Cumulative Return")
plt.legend()
plt.grid(True)
plt.show()

NameError: name 'cumulative_returns' is not defined

<Figure size 1200x600 with 0 Axes>

In [5]:
watchlist = ["RELIANCE.NS", "TCS.NS", "INFY.NS", "HDFCBANK.NS", "TATAMOTORS.NS", "ICICIBANK.NS", "SBIN.NS"]

In [6]:
if st.sidebar.button("🚀 Run Multi-Stock Scanner"):
    st.subheader("🔍 Market-Wide Opportunities")
    results = []

    with st.spinner('संपूर्ण मार्केट स्कॅन होत आहे... कृपया थांबा.'):
        for t in watchlist:
            # डेटा मिळवणे
            d = yf.download(t, period="1mo", interval="1d", progress=False)
            d['RSI'] = calculate_rsi(d['Close'])
            
            l_price = d['Close'].iloc[-1].item()
            l_rsi = d['RSI'].iloc[-1].item()
            
            # सेंटीमेंट (Stock नाव काढून)
            s_name = t.split('.')[0]
            h = get_live_news(s_name)
            s_score = sum([analyzer.polarity_scores(x)['compound'] for x in h]) / len(h) if h else 0
            
            # सिग्नल लॉजिक
            sig = "WAIT ⏳"
            if l_rsi < 40 and s_score > 0.1: sig = "BUY 🚀"
            elif l_rsi > 70 and s_score < -0.1: sig = "SELL 🔥"
            
            results.append({"Ticker": t, "Price": f"₹{l_price:.2f}", "RSI": f"{l_rsi:.1f}", "Sentiment": f"{s_score:.2f}", "Signal": sig})

    # रिझल्ट टेबल स्वरूपात दाखवणे
    res_df = pd.DataFrame(results)
    
    # हायलाईट करण्यासाठी स्टाईलिंग
    def highlight_signal(val):
        color = 'green' if val == 'BUY 🚀' else 'red' if val == 'SELL 🔥' else 'white'
        return f'background-color: {color}; color: black; font-weight: bold'

    st.table(res_df.style.applymap(highlight_signal, subset=['Signal']))

NameError: name 'st' is not defined